# Demo 07e — Meta Models: Stacking and Neural Stacking

**Class 07 · E5** (slides: `slides/07_ensembles.md`)

**Stacking** trains a meta model on the out-of-fold predictions of the six shallow models.
**Neural stacking** works on what networks learned, not on what they predict: two networks of different topology are trained on the task, their output layers are removed, both are frozen, and a small meta network reads their last hidden layers.

The networks are slow to train, so their scores are kept in `.cache/07_neural.csv`: delete that file to run them again (about four minutes on 16 cores).

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from joblib import Parallel, delayed
from sklearn.base import clone
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from threadpoolctl import threadpool_limits

from faa import boundaries, demo, metrics, plotlib
from faa import galleries as ga
from faa import synthetic as sy
from faa.demo import show, slide
from faa.galleries import CLASS_SETS, MEMBERS, REG_SETS, board, family_figure, gain_figure, names_of
from faa.metrics import mcc

demo.setup("07")
SEED = 0
N_JOBS = 8

X_tr, y_tr, X_te, y_te = sy.load("spiral", SEED)
LIM = boundaries.plane_limits(X_tr, pad=0.25)

In [ ]:
slide("Stacking: a Meta-Learner on Out-of-Fold Predictions")


class HandStack:
    """Stacking of probability outputs with a logistic-regression meta-learner.
    mode "oof":   meta features = 5-fold out-of-fold probabilities of every member (the right way);
    mode "leak":  meta features = the probabilities of members fitted on ALL training rows, predicted on the same rows;
    mode "blend": members are fitted on 70 % of the rows and the meta-learner on the other 30 % (a hold-out)."""

    def __init__(self, members, mode: str = "oof", seed: int = 0):
        self.members, self.mode, self.seed = members, mode, seed

    def _features(self, models, X):
        return np.column_stack([m.predict_proba(X)[:, 1] for m in models])

    def fit(self, X, y):
        if self.mode == "blend":
            a, b, c, d = train_test_split(X, y, test_size=0.3, stratify=y, random_state=self.seed)
            self.fitted_ = [clone(m).fit(a, c) for _, m in self.members]
            Z, t = self._features(self.fitted_, b), d
        else:
            self.fitted_ = [clone(m).fit(X, y) for _, m in self.members]
            if self.mode == "leak":
                Z = self._features(self.fitted_, X)
            else:
                cv = StratifiedKFold(5, shuffle=True, random_state=self.seed)
                Z = np.column_stack(
                    [cross_val_predict(clone(m), X, y, cv=cv, method="predict_proba")[:, 1] for _, m in self.members]
                )
            t = y
        self.meta_ = LogisticRegression().fit(Z, t)
        self.meta_features_, self.meta_target_ = Z, t
        self.classes_ = np.array([0, 1])
        return self

    def predict_proba(self, X):
        return self.meta_.predict_proba(self._features(self.fitted_, X))

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] > 0.5).astype(int)

    def score(self, X, y):
        return mcc(y, self.predict(X))


base = ga.shallow_classifiers(SEED)
MEM = [("1-NN", base["1-NN"]), ("deep tree", DecisionTreeClassifier(random_state=SEED)), ("SVM RBF", base["SVM RBF"])]
stacks = {mode: HandStack(MEM, mode, SEED).fit(X_tr, y_tr) for mode in ("leak", "oof", "blend")}
fitted_mem = {n: clone(m).fit(X_tr, y_tr) for n, m in MEM}
sk_stack = StackingClassifier(
    [(n, clone(m)) for n, m in MEM],
    final_estimator=LogisticRegression(),
    cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
).fit(X_tr, y_tr)
rows = [(n, mcc(y_tr, m.predict(X_tr)), mcc(y_te, m.predict(X_te))) for n, m in fitted_mem.items()]
rows += [(f"stack ({k})", s.score(X_tr, y_tr), s.score(X_te, y_te)) for k, s in stacks.items()]
rows.append(("StackingClassifier", mcc(y_tr, sk_stack.predict(X_tr)), mcc(y_te, sk_stack.predict(X_te))))
show(pl.DataFrame(rows, schema=["model", "train MCC", "test MCC"], orient="row"))
for mode in ("leak", "oof", "blend"):
    print(f"meta weights ({mode}) on [1-NN, deep tree, SVM]: {np.round(stacks[mode].meta_.coef_[0], 2).tolist()}")

In [ ]:
slide("The Leak: an Optimistic Score")
rows = []
for mode in ("leak", "oof"):
    Z, t = stacks[mode].meta_features_, stacks[mode].meta_target_
    cv_meta = float(
        cross_val_score(
            LogisticRegression(),
            Z,
            t,
            cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
            scoring="matthews_corrcoef",
        ).mean()
    )
    rows.append((mode, mcc(t, stacks[mode].meta_.predict(Z)), cv_meta, stacks[mode].score(X_te, y_te)))
show(pl.DataFrame(rows, schema=["meta features", "meta train MCC", "5-fold CV MCC", "test MCC"], orient="row"))

In [ ]:
# plot e5_stack
fg, axs = plt.subplots(2, 3, figsize=(plotlib.FULL, 1.85))
fg.subplots_adjust(left=0.005, right=0.995, top=0.94, bottom=0.07, hspace=0.42, wspace=0.05)
panels = [
    *fitted_mem.items(),
    ("stack, in-fold (leak)", stacks["leak"]),
    ("stack, out-of-fold", stacks["oof"]),
    ("stack, blending", stacks["blend"]),
]
for ax, (name, model) in zip(axs.ravel(), panels):
    boundaries.frontier_panel(
        ax, model, X_tr, y_tr, LIM, title=name, label=f"MCC {mcc(y_te, model.predict(X_te)):.2f}", fontsize=6.2
    )
plotlib.save(fg, "e5_stack")

## Sweep: the leak against the strength of the meta model (no slide)

Small $C$ means strong shrinkage: the meta model cannot over-trust a member.

In [ ]:
slide("The Leak Against the Strength of the Meta Model (no slide)")


def leak_run(seed, dataset):
    a, c, b, d = sy.load(dataset, seed)
    g = ga.shallow_classifiers(seed)
    mem = [g["1-NN"], DecisionTreeClassifier(random_state=seed), g["SVM RBF"]]
    cv = StratifiedKFold(5, shuffle=True, random_state=seed)
    with threadpool_limits(1):
        fitted = [clone(x).fit(a, c) for x in mem]
        in_fold = np.column_stack([x.predict_proba(a)[:, 1] for x in fitted])
        out_fold = np.column_stack(
            [cross_val_predict(clone(x), a, c, cv=cv, method="predict_proba")[:, 1] for x in mem]
        )
        test = np.column_stack([x.predict_proba(b)[:, 1] for x in fitted])
    return [
        (dataset, C, k, mcc(d, LogisticRegression(C=C).fit(Z, c).predict(test)))
        for C in (0.01, 1.0, 100.0, 10_000.0)
        for k, Z in (("leak", in_fold), ("out-of-fold", out_fold))
    ]


rr = Parallel(n_jobs=N_JOBS)(delayed(leak_run)(s, d) for s in ga.SEEDS for d in ("spiral", "moons"))
leak = pl.DataFrame([r for run in rr for r in run], schema=["dataset", "C", "features", "MCC"], orient="row")
show(
    leak.group_by("dataset", "C", "features")
    .agg(pl.col("MCC").mean())
    .pivot(on="features", index=["dataset", "C"], values="MCC")
    .sort("dataset", "C")
)

## Neural stacking

Step by step, in Keras:

1. Build NN A and NN B with different hidden layers. Each `member` call returns the full network and the same network cut below its output layer.
2. Train both on 80 % of the training set; the other 20 % stops the training early.
3. Freeze both (`trainable = False`).
4. Concatenate their last hidden layers and add one hidden layer and an output: this is the head, and the only part `fit` can change.

Planes and curves use ReLU (128, 128, 128) and GELU (64, 64, 64, 64). Tables use tanh (256, 64) and swish (128, 128, 32). The head has 32 units in every case. Regression trains with squared error and reports MAE.

In [ ]:
LOCAL = ([(128, "relu")] * 3, [(64, "gelu")] * 4)
TABLE = ([(256, "tanh"), (64, "tanh")], [(128, "swish"), (128, "swish"), (32, "swish")])


def member(n_inputs, layers, task):
    """A network and the same network without its output layer."""
    import keras

    inputs = keras.Input((n_inputs,))
    hidden = inputs
    for units, activation in layers:
        hidden = keras.layers.Dense(units, activation=activation, kernel_regularizer=keras.regularizers.L2(1e-5))(
            hidden
        )
    output = keras.layers.Dense(1, activation="sigmoid" if task == "c" else None)(hidden)
    return keras.Model(inputs, output), keras.Model(inputs, hidden)


def train(model, split, task):
    import keras

    model.compile(keras.optimizers.Adam(2e-3), "binary_crossentropy" if task == "c" else "mse")
    stop = keras.callbacks.EarlyStopping("val_loss", patience=40, restore_best_weights=True)
    slow = keras.callbacks.ReduceLROnPlateau("val_loss", factor=0.5, patience=15)
    X, y, X_val, y_val = split
    model.fit(X, y, validation_data=(X_val, y_val), epochs=300, batch_size=32, verbose=0, callbacks=[stop, slow])


class NeuralStack:
    def __init__(self, task, topology, seed):
        self.task, self.topology, self.seed = task, topology, seed

    def fit(self, X, y):
        import keras
        from keras import layers

        keras.utils.set_random_seed(self.seed)
        self.scaler = StandardScaler().fit(X)
        X = self.scaler.transform(X)
        self.mean, self.sd = (float(y.mean()), float(y.std())) if self.task == "r" else (0.0, 1.0)
        t = (y - self.mean) / self.sd
        order = np.random.default_rng(self.seed).permutation(len(X))
        fit, val = order[: int(0.8 * len(X))], order[int(0.8 * len(X)) :]
        split = X[fit], t[fit], X[val], t[val]
        self.nn_a, body_a = member(X.shape[1], self.topology[0], self.task)
        self.nn_b, body_b = member(X.shape[1], self.topology[1], self.task)
        train(self.nn_a, split, self.task)
        train(self.nn_b, split, self.task)
        body_a.trainable = body_b.trainable = False  # frozen: only the head learns from here on
        inputs = keras.Input((X.shape[1],))
        h = layers.Concatenate()([body_a(inputs), body_b(inputs)])
        h = layers.Dense(32, activation="relu", kernel_regularizer=keras.regularizers.L2(1e-3))(h)
        self.head = keras.Model(inputs, layers.Dense(1, activation="sigmoid" if self.task == "c" else None)(h))
        train(self.head, split, self.task)
        return self

    def view(self, which):
        return View(self, {"NN A": self.nn_a, "NN B": self.nn_b, "neural stack": self.head}[which])


class View:
    """One of the three networks, with the predict and predict_proba of a scikit-learn model."""

    def __init__(self, stack, net):
        self.stack, self.net = stack, net

    def raw(self, X):
        return (
            self.net.predict(self.stack.scaler.transform(X), verbose=0, batch_size=4096).ravel() * self.stack.sd
            + self.stack.mean
        )

    def predict(self, X):
        return (self.raw(X) > 0.5).astype(int) if self.stack.task == "c" else self.raw(X)

    def predict_proba(self, X):
        p = self.raw(X)
        return np.c_[1 - p, p]


def topology_of(dataset):
    return TABLE if sy.kind_of(dataset) == "table" else LOCAL


def neural_run(dataset, seed):
    os.environ["KERAS_BACKEND"] = "jax"
    os.environ["XLA_FLAGS"] = "--xla_cpu_multi_thread_eigen=false intra_op_parallelism_threads=1"
    task = sy.task_of(dataset)
    X_tr, y_tr, X_te, y_te = sy.load(dataset, seed)
    stack = NeuralStack(task, topology_of(dataset), seed).fit(X_tr, y_tr)
    return [(dataset, m, seed, metrics.task_score(task, y_te, stack.view(m).predict(X_te))) for m in ga.NEURAL]

In [ ]:
slide("Neural Stacking: the Board")
CACHE = plotlib.ROOT / ".cache" / "07_neural.csv"
if CACHE.exists():
    nn = pl.read_csv(CACHE)
else:
    runs = Parallel(n_jobs=-1)(delayed(neural_run)(d, s) for d in [*CLASS_SETS, *REG_SETS] for s in ga.SEEDS)
    nn = pl.DataFrame([r for run in runs for r in run], schema=["dataset", "model", "seed", "score"], orient="row")
    CACHE.parent.mkdir(exist_ok=True)
    nn.write_csv(CACHE)
F = list(ga.FAMILIES["c"])
clf = pl.concat([ga.evaluate(CLASS_SETS, names_of("c", F)), nn.filter(pl.col("dataset").is_in(CLASS_SETS))])

In [ ]:
reg = pl.concat([ga.evaluate(REG_SETS, names_of("r", F)), nn.filter(pl.col("dataset").is_in(REG_SETS))])

In [ ]:
all_c = [m for ms in ga.FAMILIES["c"].values() for m in ms]
all_r = [m for ms in ga.FAMILIES["r"].values() for m in ms]
show(ga.wide(clf, all_c, CLASS_SETS))
show(ga.wide(reg, all_r, REG_SETS))
others_c = [m for m in all_c if m != "neural stack"]
others_r = [m for m in all_r if m != "neural stack"]
gain_c = ga.gain(clf, CLASS_SETS, ["neural stack"], others_c, "c")
gain_r = ga.gain(reg, REG_SETS, ["neural stack"], others_r, "r")
show(gain_c)
show(gain_r)
print("the neural stack against its two members:")
show(ga.gain(clf, CLASS_SETS, ["neural stack"], ga.NEURAL[:2], "c"))
show(ga.gain(reg, REG_SETS, ["neural stack"], ga.NEURAL[:2], "r"))
print("winner of each data set:")
for res, task in ((clf, "c"), (reg, "r")):
    means = res.group_by("dataset", "model").agg(pl.col("score").mean()).sort("score", descending=task == "c")
    show(means.group_by("dataset", maintain_order=True).first().sort("dataset"))

In [ ]:
plotlib.save(board(clf, "c", CLASS_SETS), "e5_board_c")

In [ ]:
plotlib.save(board(reg, "r", REG_SETS), "e5_board_r")

In [ ]:
plotlib.save(gain_figure(gain_c, "gain in MCC: neural stack over the best other model"), "e5_gain_c")

In [ ]:
plotlib.save(gain_figure(gain_r, "gain in MAE: neural stack over the best other model"), "e5_gain_r")

In [ ]:
def full_gallery(dataset):
    task = sy.task_of(dataset)
    X_tr, y_tr, *_ = sy.load(dataset, SEED)
    fam = ga.FAMILIES[task]
    rows = {
        "shallow": fam["shallow"],
        "voting": fam["voting"],
        "trees": fam["bagging"] + fam["boosting"],
        "meta": fam["meta"],
    }
    stack = NeuralStack(task, topology_of(dataset), SEED).fit(X_tr, y_tr)
    notes = {"voting": MEMBERS[task], "meta": "A: ReLU 128 x 3\nB: GELU 64 x 4\nhead: 32 units"}
    return family_figure(dataset, rows, notes, {m: stack.view(m) for m in ga.NEURAL})

In [ ]:
plotlib.save(full_gallery("spiral"), "e5_gallery_spiral")

In [ ]:
plotlib.save(full_gallery("checkerboard"), "e5_gallery_checker")

In [ ]:
plotlib.save(full_gallery("sawtooth"), "e5_gallery_saw")

In [ ]:
plotlib.save(full_gallery("chirp"), "e5_gallery_chirp")